# 3.4.1 Data Cleaning

Injecting and imputing missing values, outlier removal via IQR on California Housing.

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer,KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
h=fetch_california_housing(); X,y=h.data.astype(float).copy(),h.target
rng=np.random.default_rng(42); mask=rng.random(X.shape)<.1; X[mask]=np.nan
print('Missing per feature:',np.isnan(X).sum(0))

In [ ]:
Xt,Xv,yt,yv=train_test_split(X,y,test_size=.2,random_state=42)
for strat in ['mean','median']:
    p=make_pipeline(SimpleImputer(strategy=strat),StandardScaler(),Ridge(1.))
    p.fit(Xt,yt); print(f'{strat}: RMSE={mean_squared_error(yv,p.predict(Xv))**.5:.4f}')
p=make_pipeline(KNNImputer(5),StandardScaler(),Ridge(1.)); p.fit(Xt,yt)
print(f'KNN5: RMSE={mean_squared_error(yv,p.predict(Xv))**.5:.4f}')

In [ ]:
# IQR outlier removal
Xo,yo=h.data,h.target; Q1,Q3=np.percentile(Xo,25,0),np.percentile(Xo,75,0)
mask=((Xo>=Q1-3*(Q3-Q1))&(Xo<=Q3+3*(Q3-Q1))).all(1)
print(f'Rows after IQR(3x): {mask.sum()} (removed {(~mask).sum()})')
Xc,yc=Xo[mask],yo[mask]
Xt2,Xv2,yt2,yv2=train_test_split(Xc,yc,test_size=.2,random_state=42)
p=make_pipeline(StandardScaler(),Ridge(1.)); p.fit(Xt2,yt2)
print(f'Ridge cleaned: RMSE={mean_squared_error(yv2,p.predict(Xv2))**.5:.4f}')